# Projekt końcowy z laboratorium “Wstęp do bioinformatyki 1” (2025/26)
(max 36 punktów)


## Część I - Analiza danych w Pythonie
Przeprowadź analizę danych na podstawie publikacji [[Harigaya & Parker 2016]](https://link.springer.com/article/10.1186/s12864-016-3237-6#Sec15).

### 1. Pobieranie danych

* a. Zapisać na dysku tabelę “Additional file 22: Table S4. - plik powinien zostać pobrany po kliknięciu w link.



In [ ]:
# Prepare assignment-local inputs.
from pathlib import Path
data_folder = Path("data")
file_name = "part_1.xls"
data_file = "csv_data.csv"
!python3 download_data.py


* b. Zapoznaj się z pobranym plikiem, a następnie zapisz dane z karty “Table” w formacie csv.
  * Jednym ze sposobów jest otwarcie zapisanego pliku w programie Excel lub
równoważnym (np. w Google Drive), wybranie zakładki “Table”i zapisanie jej
w formacie CSV poprzez opcję "Zapisz jako".

In [ ]:
!pip install pyexcel pyexcel-xls
import pyexcel as pe

# zapisujemy dane w formacie csv
pe.save_book_as(file_name=f"{data_folder}/{file_name}", dest_file_name=f"{data_folder}/{data_file}")

### 2. Przygotowanie danych:

* a. Proszę wczytać dane w formacie csv do Pythona (z wykorzystaniem Pandas lub
Polars). Uwaga: Proszę zwrócić uwagę na separator kolumn - jeśli w trakcie zapisu
danych do pliku csv został wybrany tabulator jako separator, należu do uwzględnić
podczas wczytywania danych (np. dodając odpowiedni argument “sep”).

In [ ]:
# wersja: ładowanie bezpośrednie z .xls
## Biblioteki
import polars as pl
import pandas as pd
import plotly.express as px

!pip install fastexcel

## paths
data_folder = Path("data")
file_name = 'part_1.xls'

## loading data
data = pl.read_excel(f"{data_folder}/{file_name}", sheet_name='Table')


In [ ]:
# wersja: ładowanie z .csv
import polars as pl
import pandas as pd
import plotly.express as px

# wczytujemy dane z karty Tabele w formacie csv do Pythona
data_file = data_file[:8]
data = pl.read_csv(f"{data_folder}/{data_file}__Table__2.csv", separator=",")

# konwerujemy wszystkie kolumny (oprócz kolumny "Gene ID")  na liczby zmiennoprzecinkowe (Float64)
data = data.with_columns([
    pl.col(c).cast(pl.Float64, strict=False) for c in data.columns if c != "Gene ID"
])


* b. Proszę wyświetlić początek tabeli odpowiednim poleceniem i upewnić się, że dane zostały poprawnie wczytane.


In [ ]:
data.head()

* c. Proszę wypisać listę wszystkich kolumn w tabeli, aby zidentyfikować interesujące zmienne.

In [ ]:
# lista kolumn
data.columns

In [ ]:
# mapa kolumn
data.schema

* d. Proszę zachować tylko kolumny: "Gene ID" (identyfikator genu) oraz wszystkie
kolumny rozpoczynające się od "HL" (okresy półtrwania, ang. half-life) - należy
usunąć wszystkie pozostałe kolumny.

In [ ]:
data_subset = data.select(
    [col_name for col_name in data.columns
     if col_name == "Gene ID" or col_name.startswith("HL")
    ])
data_subset


* e. W ramach dalszej filtracji, proszę usunąć wszystkie wiersze zawierające wartości
NaN/ NA.

In [ ]:
data_subset = data_subset.drop_nans().drop_nulls()
data_subset

### 3. Analiza danych w kierunku identyfikacji najmniej i najbardziej stabilnych transkryptów:


* a. Tabela zawiera wyniki stabilności transkryptów uzyskane z różnych eksperymentów.
Przed przystąpieniem do pracy nad właściwą analizą danych proszę zdefiniować na
podstawie literatury i zapisać w raporcie co oznacza “half- life” w kontekście
stabilności mRNA.


<details>
<summary>Ważne cytaty</summary>

1. mRNA synthesis rates can be measured by multiple methods including genomic run-on (GRO) [39]. Direct measurement of mRNA half-lives is performed either by transcription shutoff or metabolic labeling. Although transcription shutoff is usually achieved by an addition of transcription inhibitor, in S. cerevisiae, many studies have used the rpb1-1 temperature sensitive mutant of the RBP1 gene, encoding the largest subunit of RNA polymerase II [40]. An alternative to the rbp1-1 strain is the rbp1-frb strain, in which Rbp1 is depleted from the nucleus upon an addition of the drug rapamycin via the “anchor-away” technique [8, 41]. Metabolic labeling using 4-thiouridine (4sU) or 4-thiouracil (4tU) can measure mRNA half-lives with minimal perturbation [42–47]. In one scheme, time course experiments are performed after an addition of 4sU/4tU to cell cultures [44, 46–48], which we refer to as “4sU/4tU labeling.” In another scheme, samples are analyzed after an addition of Uracil to cultures that have been grown in the presence of 4sU/4tU [45], which we refer to as “4sU/4tU chase” (Table 1).

2. We used “minute,” “molecule per minute per cell,” and “molecule per cell” as units of mRNA half-lives, synthesis rates, and abundance, respectively. When the values were expressed otherwise in the source data, we converted them accordingly. In our calculations, we used a total number of mRNAs per cell of 60,000 and that of 41,000 for S. cerevisiae and S. pombe, respectively [72, 74]. When mRNA synthesis rates were not available in the source data, based on the steady-state assumption [75], we computed them from mRNA half-lives and abundance using the equations λ = log2/hl and μ = m (α + λ), where λ is the mRNA decay rate [min−1], hl is the mRNA half-life [min], μ is the mRNA synthesis rate [RNA min−1 cell−1], m is the mRNA abundance [cell−1], and α is the cell growth rate [min−1]. When mRNA abundance was not available in the source data, we used data obtained by Ito and colleagues [71] and those obtained by Bahler and colleagues [72] for S. cerevisiae and S. pombe, respectively. For the datasets in S. cerevisiae, we used mRNA abundance obtained in matched types of culture media. That is, we used mRNA abundance in the rich nutrient YPD medium for the “Brown (1),” “Brown (2),” “Hughes,” “Peltz,” “Pilpel,” “Struhl,” “Coller (1),” and “Coller (2)” datasets, whereas we used that in the minimal nutrient SC medium for the “Weis” dataset (Additional file 21: Table S3). For mRNA half-live data obtained via the rbp1-1 and rbp1-frb systems, where cell growth is halted, or those obtained without taking into account cell division, we set α to zero. For the “Weis” data, we computed α from a cell cycle length of 150 min using the equation α = log2/ccl, where ccl is the cell cycle length [min].
</details>

#### Okres półtrwania cząsteczki mRNA (ang. mRNA half-life)

* jednostka: minuta [min] lub cząsteczka na minutę na komórkę lub cząsteczka na komórkę (cytat: "We used “minute”, “molecule per minute per cell” and “molecule per cell” as units of mRNA half-lives")
* oznaczenia: $hl$ (np. w równaniu $λ=log2/hl$)
* metoda pomiaru: znakowanie metaboliczne (ang. metabolic labeling)
(cytat: "Metabolic labeling using 4-thiouridine (4sU) or 4-thiouracil (4tU) can measure mRNA half-lives with minimal perturbation [42–47].")

Im większa warotość $hl$ (czyli im dłuższy okres półtrwania), tym bardziej stabilna cząsteczka mRNA.

In [ ]:
data_subset.schema

* b. Statystyki opisowe: Dla każdego zbioru danych (“HL-Mata (1)”, “HL- Mata (2)”, itd.)
proszę policzyć: wartość minimalną, średnią, maksymalną oraz medianę.
Proszę umieścić te wartości w raporcie.


In [ ]:
data_subset.drop(['Gene ID']).unpivot().group_by("variable").agg([
    pl.min('value').alias('min'),
    pl.mean('value').alias('mean'),
    pl.max('value').alias('max'),
    pl.median('value').alias('median'),
    (pl.quantile('value', 0.75) - pl.quantile('value', 0.25)).alias('IQR')
])

* c. Porównanie zbiorów danych: Proszę porównać rozkład parametrów HL pomiędzy
zbiorami danych, również dla eksperymentów tej samej grupy badawczej (w tabeli
znajduje się 5 zbiorów danych z różnych eksperymentów grupy prof. Juana Maty).
  * i. Wizualizacja danych: proszę przygotować wykres typu box plot lub podobny
(boxen plot, violin plot), w którym na osi X znajdą się nazwy zbiorów danych,
natomiast na osi Y - wartości numeryczne HL (“half-life”). Proszę podpisać
osie na wykresie. Jeśli to konieczne, proszę przekształcić tabelę przed wykonaniem wykresu, np. za pomocą melt().

      * Polecenie należy zaktualizować dla polaris: .melt -> .unpivot

#### Analiza rozkładów HL

##### Cel
Celem analizy jest porównanie rozkładów czasów półtrwania (HL) mRNA pomiędzy zbiorami danych wygenerowanymi metodami 4sU/4tU, z uwzględnieniem wpływu konfiguracji eksperymentalnej oraz modelu matematycznego na położenie oraz kształt rozkładu.
##### Wstęp biologiczny
Działanie znakowania 4sU/4tU:
- 4sU (4-thiouridine) lub 4tU (4-thiouracil) są analogami urydyny/uracylu, w których atom tlenu na pozycji 4 został zastąpiony siarką.
- Po dodaniu substancji do hodowli nowo syntetyzowane RNA inkorporuje znacznik.

Na podstawie zmian ilości znakowanego RNA w czasie dopasowywany jest model kinetyczny, pozwalający oszacować tempo syntezy oraz degradacji transkryptów.

Ważny w całej metodzie jest brak globalnego wyłączenia transkrypcji, zatem:
- brak odpowiedzi stresowej.
Autorzy sugerują, że 4sU/4tU bez selekcji oligo(dT) najwierniej oddaje fizjologię komórki.

(fragment:)
"Although it is not possible to rigorously determine which datasets are most biologically accurate, we suggest that the results obtained via 4sU/4tU labeling without oligo(dT) selection (“Cramer (1)”, “Cramer (2),” and “Gresham”) are more likely to reflect the physiological nature of gene regulation for the following reasons."
##### Wyniki
Rozkłady HL mają zbliżony kształt, z wyjątkiem `HL - Mata (3)` oraz `HL - Cramer`, dla których wartość IQR jest większa z powodu wartości odstających.

Widzimy jednak różnice, jeżeli chodzi o ich położenie. Wskazują na to różne mediany oraz średnie.
##### Wnioski


Różnice w położeniu rozkładów (mediana, średnia) wynikają głównie z różnic w czasie znakowania oraz liczbie punktów czasowych. Pomimo różnic w skali absolutnej, można się spodziewać występowania korelacji, co można zaobserować na wykresach z punktu d, co jest wykazane w artykule. W przypadku zbioru `HL - Gagneur` obecność bardzo małych wartości HL wynika ze stosowania innego modelu matematycznego, który rozdziela:
- transkrypcję pre-mRNA,
- splicing mRNA,
- degradację każdej frakcji osobno,

co powoduje, że sygnał który był przypisywany do stabilności mRNA zostaje przypisany do opóźnień splicingowych. Może to sugerwać błędy w innych pracach które przypisują stabilność RNA do długiego okresu bycia nie splice'owanym i dużym tempem rozpadu $\lambda$, co prowadzi do fałszywych wniosków.

(fragment:)
"half-lives in the different datasets showed moderate to strong correlations … however, the absolute values differed substantially … These observations represent the technical challenges in quantifying absolute kinetic values but suggest that relative values are reasonably reliable."

In [ ]:
fig = px.box(data_subset.drop(['Gene ID']).unpivot() , y='value', color='variable',
          labels = {'value': 'HL', 'variable': 'Zbiór danych'})
fig.update_layout(
    title='log hf distribution',
    yaxis=dict(
        title='log hf distribution',
        type='log',              # zmiana skali
    ),
    xaxis_title = 'Zbiór danych',
    legend_title = 'Zbiór danych'
)


* d. Porównanie pary zbiorów - wykresy punktowe:
  * i. Proszę przygotować wykres punktowy (scatter plot), w którym zostaną
porównane wartości HL (half-life) pomiędzy odpowiednimi zbiorami danych - na osiach X i Y powinny znaleźć się odpowiednio:
    1. Wykres 1: Porównanie “HL-Mata (1)” i “HL- Mata (2)”.
    2. Wykres 2: Porównanie “HL - Mata (5)” i “HL-Gagneur”
  * ii. Biorąc pod uwagę, że każdy z punktów na wykresie powinien
reprezentować jeden gen z tabeli, proszę odpowiedzieć na pytanie:
Czy taki rodzaj reprezentacji danych umożliwia wstępne porównanie
eksperymentów lub identyfikację potencjalnych wartości odstających (ang.
outliers)?
  * iii. Jeśli na powyższe pytanie odpowiedzieli Państwo twierdząco i
zaobserwowali Państwo występowanie potencjalnych wartości odstających,
proszę przefiltrować tabelę i usunąć te dane z dalszej analizy (np. poprzez
odrzucenie wartości mniejszych lub większych niż zadana wartość).

#### Porównanie pary zbiorów (podpunkt d)
##### ii
Tak, widzimy, że wartość HL będzie się różnić pomiędzy eksperymentami, jednak powinna między nimi istnieć zależność liniowa.
##### iii.
Wykorzystując informacje o liniowości, zastosujemy następującą metodę:
1. Wyliczenie reszty
$$e_i = y_i = \hat{y}_i$$
2. Wyliczenie dźwigni
$$h_{ii} = \mathrm{diag}(H), H = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$$
3. Wyliczenie residuów:
$$r_i= \frac{e_i}{s_{(i)}\sqrt{1 - h_{ii}}}$$
4. Porównanie z rozkładem t-studenta
$$r_i \sim t_{n-p-1}$$
5. Zastosowanie korekty Bonferroniego:
$$p_i^{(\mathrm{Bonf})}=\min{(n\cdot p_i, 1)}$$
6. Odrzucenie wartości krytycznych na poziomie istotności $\alpha=0.05$
$$p_i^{(\mathrm{Bonf})} <\alpha$$

W celu filtracji wybieramy ostatecznie dwa zbiory `HL - Mata (5)` oraz `HL - Gagneur`, otrzymany wynik mamy w zmiennej `result`. Ważne jest, że te outliery nie muszą się pokrywac pomiędzy innymi porównaniami.

Ostatecznie do analizy wybieramy zbiór `HL - Gagneur`, z powodu modelu kinetycznego który uwzględnia opóźnienia splicingowe. Odrzucamy geny które uznaliśmy za istotne statystycznie outliery

In [ ]:
data_subset.schema

In [ ]:
# Plot 1.
px.scatter(data_subset, x='HL - Mata (1)', y='HL - Mata (2)').show()

# Plot 2.
px.scatter(data_subset, x='HL - Mata (5)', y='HL - Gagneur').show()

In [ ]:
data_subset.schema

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
def verify_outliers(data_subset: pl.DataFrame, var1: str, var2: str, alpha=0.05):
    '''
    Zwraca zbiór danych z oznaczonymi wartościami odstającymi na poziomie `alpha`
    Zakładamy istnienie zależności liniowej pomiędzy zmiennymi `var1` i `var2`
    '''
    # dodanie stałego indeksu numerycznego
    if 'row_id' not in data_subset.columns:
        data_subset_idx = data_subset.with_row_index("row_id")
    else:
        data_subset_idx = data_subset

    # wytrenowanie modelu
    X = sm.add_constant(data_subset_idx.select([var1]))
    y = data_subset_idx.select([var2]).to_numpy()
    model = sm.OLS(y, X).fit()

    # Testy
    ## wyliczenie residuów
    r = model.get_influence().resid_studentized_external
    ## Stopnie swobody
    n = X.shape[0] # liczba obserwacji
    p = X.shape[1] # liczba zmiennych
    df_t = n - p -1
    ## p_values
    p_values = 2 * (1 - stats.t.cdf(np.abs(r), df=df_t))

    # zestawienie wyników
    ## zebranie informacji
    results = pl.DataFrame({
        'row_id': data_subset_idx['row_id'],
        'studentized_residual': r,
        'p_value': p_values
    })

    ## Korekta Bonferroniego
    results = results.with_columns(
        pl.when(pl.col("p_value") * n > 1.0)
        .then(1.0)
        .otherwise(pl.col("p_value") * n)
        .alias("p_bonf")
    )
    results = results.with_columns(
        (pl.col('p_bonf') < alpha).alias(f'reject_bonf_05pct_{var1}_{var2}')
    )

    ## złączenie wyników
    results = data_subset_idx.join(results, on='row_id', how='left')

    # ## odrzucenie wyników
    # na ten moment chowam, żeby móc wykresy pokazać #
    # results = results.filter(pl.col('reject_bonf_05pct')==False)

    ## zwrot wyników
    return results.drop(['studentized_residual', 'p_value'])


In [ ]:
results_0 = verify_outliers(data_subset, 'HL - Mata (1)', 'HL - Mata (2)')
result = verify_outliers(results_0, 'HL - Mata (5)', 'HL - Gagneur')

In [ ]:
# wykresy z oznakowanymi odchyleniami
## Plot 1.
px.scatter(result, x='HL - Mata (1)', y='HL - Mata (2)', color='reject_bonf_05pct_HL - Mata (1)_HL - Mata (2)').show()

## Plot 2.1
px.scatter(result, x='HL - Mata (5)', y='HL - Gagneur', color='reject_bonf_05pct_HL - Mata (5)_HL - Gagneur').show()
# ## Plot 2.2 - ten wykres pokazuje że zmiana jest dobra tylk względem dwóch datasetów (pomiędzy innymi występuja inna monotoniczność)
# px.scatter(result, x='HL - Mata (1)', y='HL - Mata (2)', color='reject_bonf_05pct_HL - Mata (5)_HL - Gagneur').show()

* e. Wyszukiwanie genów o skrajnych wartościach “half-life”
Proszę wybrać jeden ze zbiorów danych (podać jego nazwę w raporcie) i wypisać:
    1. 10 genów o najniższych wartościach half-life,
    2. 10 genów o najwyższych wartościach half-life.

      * Można skorzystać np. z funkcji sort().
  * i. Proszę wyszukać te geny w wybranej bazie danych (np. PomBase). i krótko opisać ich funkcje:
    1. Czy kodują białka? Jakie?
    2. Czy ich funkcje są podobne, czy zupełnie różne?
  * ii. Wyniki proszę przedstawić w raporcie w formie tabelarycznej i opisowej.

In [ ]:
# odrzucamy kolumny które są outlierami
filt_e = result.filter(pl.col('reject_bonf_05pct_HL - Mata (5)_HL - Gagneur')==False)
# wybrany zbiór genów
gene_set = 'HL - Gagneur'

data_gene = filt_e['Gene ID' ,gene_set].sort(by=gene_set, descending=True)
# print(data_gene.head(10))
# print(data_gene.tail(10))

In [ ]:
# 10 genów o najniższych wartościach half-life

print(data_gene.tail(10))

<details>
<summary>Notatki - Funkcje 10 genów o najniższych wartościach half-life</summary>

Funkcje 10 genów o najniższych wartościach half-life

**SPBC19C2.11c**
1. Produkt ekspresji genu (białko): **ERMES complex subunit, membrane tether and lipid transfer protein Mdm34**
2. Funkcje białka:
- mitochondrion-endoplasmic reticulum membrane tether activity
- phospholipid transfer activity
- lipid binding
3. Procesy biologiczne z udziałem białka:
- intermembrane phospholipid transfer
- mitochondrial membrane organization
- mitochondrion-endoplasmic reticulum membrane tethering

**SPCC830.05c**
1. Produkt ekspresji genu (białko): **NuA4 histone acetyltransferase core complex-Piccolo HAT module linker protein Epl1**
2. Funkcje białka:
- protein-macromolecule adaptor activity
3. Procesy biologiczne z udziałem białka:
- DNA repair-dependent chromatin remodeling
- regulation of transcription by RNA polymerase II

**SPAC644.06c**
1. Produkt ekspresji genu (białko): **serine/threonine protein kinase (NIM1 family) Cdr1/Nim1**
2. Funkcje białka:
- protein serine/threonine kinase activity
  - phosphorylates cdr1 during cellular hyperosmotic salinity response
  - phosphorylates wee1 in Cdr2 medial cortical node complex
  - phosphorylates wee1 in medial cortical node during mitotic G2 phase
  - phosphorylates wee1 involved in mitotic G2 cell size control checkpoint signaling
  - phosphorylates wee1 involved in positive regulation of G2/M transition of mitotic cell cycle
- protein serine/threonine/tyrosine kinase activity (phosphorylates cdr1)
- ATP binding
- protein binding (binds msp1, nif1)
3. Procesy biologiczne z udziałem białka:
- positive regulation of G2/M transition of mitotic cell cycle
- signaling

**SPBC23E6.03c**
1. Produkt ekspresji genu (białko): **protein N-terminal amidase Nta1**
2. Funkcje białka:
- protein-N-terminal asparagine amidohydrolase activity
- protein-N-terminal glutamine amidohydrolase activity
3. Procesy biologiczne z udziałem białka:
- protein catabolic process

**SPAC29B12.08**
1. Produkt ekspresji genu (białko): **Clr5 protein**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka:
- silent mating-type cassette heterochromatin formation

**SPBC32C12.02**
1. Produkt ekspresji genu (białko): **DNA-binding transcription factor Ste11**
2. Funkcje białka:
- DNA-binding transcription activator activity, RNA polymerase II-specific
   - activates transcription of dmc1, mei2, rad24, rgs1, ste11
   - activates transcription of mfm1 at mating_type_M_box
- DNA-binding transcription repressor activity, RNA polymerase II-specific (represses transcription of fus1, map1, mei2, rep1, rgs1, sme2, sms1, spk1, ste11, ste4, ste6)
- protein binding
   - active form ste11 (Phos(T173,S218)) binds rad24
   - binds mat1-Mc, pat1, spk1
   - binds rad24 binds ste11 (Phos(T173,S218)) part of negative regulation of transcription by RNA polymerase II
- RNA polymerase II cis-regulatory region sequence-specific DNA binding (binds promoter of atg1801, byr1, csr101, gpa1, mam1, mfm2, ste4, utp8, SPAC513.04, SPNCRNA.1534)
- RNA polymerase II cis-regulatory region sequence-specific DNA binding, bending (at TR_box)
3. Procesy biologiczne z udziałem białka:
- induction of conjugation with cellular fusion (during cellular response to nitrogen starvation)
- positive regulation of induction of conjugation with cellular fusion
- positive regulation of transcription by RNA polymerase II
   - at TR_box during cellular response to nitrogen starvation
   - at TR_box during response to pheromone
   - regulates byr2, mat1-Mc, mat1-Mi, mat2-Pi, mei2, mfm1, mfm2, mfm3
   - regulates mat2-Pc during single-celled organism vegetative growth phase

**SPAPB24D3.07c**
1. Produkt ekspresji genu (białko): **glycosyl hydrolase (predicted by Pfam-N)**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka: N/A

**SPCC126.09**
1. Produkt ekspresji genu (białko): **vacuolar zinc exporter, ZIP family, Zip2**
2. Funkcje białka:
- zinc ion transmembrane transporter activity
3. Procesy biologiczne z udziałem białka:
- intracellular zinc ion homeostasis
- zinc ion export from vacuole

**SPBC25B2.08**
1. Produkt ekspresji genu (białko): **Schizosaccharomyces pombe specific protein, phosphate responsive**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka: N/A

**SPBC19F5.01c**
1. Produkt ekspresji genu (białko): **G1 cyclin Puc1**
2. Funkcje białka:
- cyclin-dependent protein serine/threonine kinase regulator activity
3. Procesy biologiczne z udziałem białka:
- negative regulation of cell cycle switching, mitotic to meiotic cell cycle
- positive regulation of G1/S transition of mitotic cell cycle
- signaling

</details>

#### Funkcje 10 genów o najniższych wartościach half-life (podpunkt e)

##### i.

###### **SPBC19C2.11c**
1. Produkt ekspresji genu (białko): **ERMES complex subunit, membrane tether and lipid transfer protein Mdm34**
2. Funkcje białka:
- transportuje fosfolipidy pomiędzy błoną mitochondiaralną i błoną siateczki śródplazmatycznej
- wiąże lipidy
3. Procesy biologiczne z udziałem białka:
- międzybłonowy transport fosfolipidów
- oragnizacja błony mitochondrialnej
- pośredniczenie pomiędzy błoną mitochondiralną i błona siateczki śródplazmatycznej

###### **SPCC830.05c**
1. Produkt ekspresji genu (białko): **NuA4 histone acetyltransferase core complex-Piccolo HAT module linker protein Epl1**
2. Funkcje białka:
- umożliwia interakcje między makrocząsteczkami
3. Procesy biologiczne z udziałem białka:
- przebudowa chromatyny zależna od naprawy DNA
- regulacja transkrypcji przez RNA polimerazę II

###### **SPAC644.06c**
1. Produkt ekspresji genu (białko): **serine/threonine protein kinase (NIM1 family) Cdr1/Nim1**
2. Funkcje białka:
- posiada aktywność kinazy serynowo-treoninowej *(kinaza fosforyluje)*
- posiada aktywność kinazy serynowo-treoninowo-tyrozynowej *(kinaza fosforyluje)*
- wiąże ATP
- wiąże białka
3. Procesy biologiczne z udziałem białka:
- pozytywna regulacja przejścia G2/M w mitotycznym cyklu komórkowym
- sygnalizacja komórkowa

###### **SPBC23E6.03c**
1. Produkt ekspresji genu (białko): **protein N-terminal amidase Nta1**
2. Funkcje białka:
- posiada aktywność N-końcowej amidohydrolazy asparaginy *(aminohydrolaza "przecina" wiązanie amidowe w wodzie)*
- posiada aktywność N-końcowej amidohydrolazy glutaminy *(aminohydrolaza "przecina" wiązanie amidowe w wodzie)*

3. Procesy biologiczne z udziałem białka:
- proces kataboliczny białek

###### **SPAC29B12.08**
1. Produkt ekspresji genu (białko): **Clr5 protein**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka:
- proces, w którym nieaktywne locus typu płciowego są wyciszane poprzez formowanie heterochromatyny

###### **SPBC32C12.02**
1. Produkt ekspresji genu (białko): **DNA-binding transcription factor Ste11**
2. Funkcje białka:
- aktywuje transkrypcję DNA (aktywator specyficzny dla RNA polimerazy II)
- hamuje transkrypcję DNA (represor specyficzny dla RNA polimerazy II)
- wiąże białka
- specyficznie, sekwencyjnie wiąże DNA w regionie regulatorowym genu (zagięcie DNA przy TR_box dla RNA polimerazy II)
- *(wiąże się specyficznie z określoną sekwencją DNA w regionach regulatorowych (promotory lub enhancery), które są „cis”, czyli znajdują się w pobliżu genu, który regulują)*
3. Procesy biologiczne z udziałem białka:
- indukcja koniugacji z fuzją komórkową (podczas odpowiedzi komórkowej na niedobór azotu)
- pozytywna regulacja indukcji koniugacji z fuzją komórkową
- pozytywna regulacja transkrypcji przez polimerazę RNA II

###### **SPAPB24D3.07c**
1. Produkt ekspresji genu (białko): **glycosyl hydrolase (predicted by Pfam-N)**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka: N/A

###### **SPCC126.09**
1. Produkt ekspresji genu (białko): **vacuolar zinc exporter, ZIP family, Zip2**
2. Funkcje białka:
- posiada katyność transbłonowego transportera jonów cynku
3. Procesy biologiczne z udziałem białka:
- wewnątrzkomórkowa homeostaza jonów cynku
- eksport jonów cynku z wakuoli

###### **SPBC25B2.08**
1. Produkt ekspresji genu (białko): **Schizosaccharomyces pombe specific protein, phosphate responsive**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka: N/A

###### **SPBC19F5.01c**
1. Produkt ekspresji genu (białko): **G1 cyclin Puc1**
2. Funkcje białka:
- posiada aktywność regulatora białkowej kinazy serynowo-treoninowej zależnej od cyklin
3. Procesy biologiczne z udziałem białka:
- negatywna regulacja przełączania cykli komórkowych (z mitotycznego do mejotycznego cyklu komórkowego)
- pozytywna regulacja przejścia G1/S w mitotycznym cyklu komórkowym
- sygnalizacja komórkowa

###### Wnioski
Funkcje białek są różne. Dwa z dziesięciu wymienionych białek biorą udział przełączaniu faz cyklu komórkowego.

In [ ]:
# 10 genów o najwyższych wartościach half-life

print(data_gene.head(10))

<details>
<summary>Notatki - Funkcje 10 genów o najwyższych wartościach half-life</summary>

Funkcje 10 genów o najwyższych wartościach half-life

**SPBC15D4.15**
1. Produkt ekspresji genu (białko): **4-phosphoerythronate phosphatase/2-phosphoglycolate phophatase, involved in detoxification of glycolytic by-products**
2. Funkcje białka
- alkaline phosphatase activity
- phosphoglycolate phosphatase activity
3. Procesy biologiczne z udziałem białka:
- cellular detoxification

**SPAC3H8.07c**
1. Produkt ekspresji genu (białko): **prefoldin subunit Pfd3**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka:
- microtubule-based process
- protein folding
- tubulin complex assembly

**SPAC1002.07c**
1. Produkt ekspresji genu (białko): **spermidine/spermine N1-acetyltransferase Sat1**
2. Funkcje białka:
- diamine N-acetyltransferase activity
3. Procesy biologiczne z udziałem białka:
- cellular detoxification of nitrogen compound
- putrescine catabolic process

**SPBC1539.07c**
1. Produkt ekspresji genu (białko): **glutathione-dependent formaldehyde dehydrogenase**
2. Funkcje białka:
- alcohol dehydrogenase (NAD+) activity
- S-(hydroxymethyl)glutathione dehydrogenase (NAD+) activity
- S-(hydroxymethyl)glutathione dehydrogenase (NADP+) activity
- S-nitrosoglutathione reductase (NADH) activity
- zinc ion binding
3. Procesy biologiczne z udziałem białka:
- formaldehyde catabolic process

**SPBC582.09**
1. Produkt ekspresji genu (białko): **peroxisomal biogenesis factor 11**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka:
- peroxisome fission

**SPCC777.12c**
1. Produkt ekspresji genu (białko): **ER stress response thioredoxin family membrane disulfide oxidoreductase**
2. Funkcje białka:
- disulfide oxidoreductase activity
3. Procesy biologiczne z udziałem białka:
- protein folding in endoplasmic reticulum
- response to endoplasmic reticulum stress

**SPAC19A8.15**
1. Produkt ekspresji genu (białko): **tryptophan synthase**
2. Funkcje białka:
- tryptophan synthase activity
3. Procesy biologiczne z udziałem białka:
- L-tryptophan biosynthetic process

**SPBC16C6.08c**
1. Produkt ekspresji genu (białko): **ubiquinol-cytochrome-c reductase complex subunit 8, hinge protein Qcr6**
2. Funkcje białka:
- quinol-cytochrome-c reductase activity
3. Procesy biologiczne z udziałem białka:
- mitochondrial electron transport, ubiquinol to cytochrome c

**SPAP7G5.04c**
1. Produkt ekspresji genu (białko): **aminoadipate-semialdehyde dehydrogenase**7
2. Funkcje białka:
-L-aminoadipate-semialdehyde dehydrogenase [NAD(P)+] activity
- phosphopantetheine binding
3. Procesy biologiczne z udziałem białka:
- L-lysine biosynthetic process via aminoadipic acid


**SPAC16E8.15**
1. Produkt ekspresji genu (białko): **translation initiation factor eIF4E, 4F complex subunit**
2. Funkcje białka:
- translation initiation factor activity
- protein binding (binds tif471)
- RNA 7-methylguanosine cap binding
3. Procesy biologiczne z udziałem białka:
- cytoplasmic translational initiation
- nuclear-transcribed mRNA catabolic process, nonsense-mediated decay

</details>

#### Funkcje 10 genów o najwyższych wartościach half-life (podpunkt e)

##### i.

###### **SPBC15D4.15**
1. Produkt ekspresji genu (białko): **4-phosphoerythronate phosphatase/2-phosphoglycolate phophatase, involved in detoxification of glycolytic by-products**
2. Funkcje białka:
- aktywność fosfatazy alkalicznej *(fosfataza defosforyluje, hydroliza)*
- aktywność fosfatazy fosfoglikolowej *(fosfataza defosforyluje, hydroliza)*
3. Procesy biologiczne z udziałem białka:
- detoksykacja komórkowa

###### **SPAC3H8.07c**
1. Produkt ekspresji genu (białko): **prefoldin subunit Pfd3**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka:
- proces oparty na mikrotubulach
- fałdowanie białek
- składanie kompleksu tubuliny

###### **SPAC1002.07c**
1. Produkt ekspresji genu (białko): **spermidine/spermine N1-acetyltransferase Sat1**
2. Funkcje białka:
- posiada aktywność N-acetylotransferazy diaminowej
3. Procesy biologiczne z udziałem białka:
- detoksykacja związków azotowych w komórce
- proces kataboliczny putrescyny

###### **SPBC1539.07c**
1. Produkt ekspresji genu (białko): **glutathione-dependent formaldehyde dehydrogenase**
2. Funkcje białka:
- posiada aktywność dehydrogenazy alkoholowej (NAD⁺)
- posiada aktywność dehydrogenazy S-(hydroksymetylo)glutationowej (NAD⁺)
- posiada aktywność dehydrogenazy S-(hydroksymetylo)glutationowej (NADP⁺)
- posiada aktywność reduktazy S-nitrozoglutationowej (NADH)
- wiąże jony cynku
3. Procesy biologiczne z udziałem białka:
- proces kataboliczny formaldehydu

###### **SPBC582.09**
1. Produkt ekspresji genu (białko): **peroxisomal biogenesis factor 11**
2. Funkcje białka: N/A
3. Procesy biologiczne z udziałem białka:
- podział peroksysomów *(m.in. detoksykacja nadtlenku wodoru, odpowiedż na stres oksydacyjny)*

###### **SPCC777.12c**
1. Produkt ekspresji genu (białko): **ER stress response thioredoxin family membrane disulfide oxidoreductase**
2. Funkcje białka:
- posiada aktywność oksydoreduktazy disiarczkowej
3. Procesy biologiczne z udziałem białka:
- fałdowanie białek w siateczce śródplazmatycznej
- odpowiedź siateczki śródplazmatycznej na stres

###### **SPAC19A8.15**
1. Produkt ekspresji genu (białko): **tryptophan synthase**
2. Funkcje białka:
- posiada aktywność syntazy tryptofanu tryptophan synthase activity
3. Procesy biologiczne z udziałem białka:
- biosynteza L-tryptofanu

###### **SPBC16C6.08c**
1. Produkt ekspresji genu (białko): **ubiquinol-cytochrome-c reductase complex subunit 8, hinge protein Qcr6**
2. Funkcje białka:
- posiada aktywność reduktazy chinol–cytochrom c
3. Procesy biologiczne z udziałem białka:
- mitochondrialny transport elektronów z ubichinolu na cytochrom c

###### **SPAP7G5.04c**
1. Produkt ekspresji genu (białko): **aminoadipate-semialdehyde dehydrogenase**7
2. Funkcje białka:
- aktywność dehydrogenazy L-aminoadypinowego półaldehydu [NAD(P)⁺]
- wiązanie fosfopanteteiny
3. Procesy biologiczne z udziałem białka:
- biosynteza L-lizyny z udziałem kwasu aminoadypinowego

###### **SPAC16E8.15**
1. Produkt ekspresji genu (białko): **translation initiation factor eIF4E, 4F complex subunit**
2. Funkcje białka:
- posiada aktywność czynnika inicjacji translacji
- wiąże białka (wiąże Tif471)
- wiąże czapeczkę 7-metyloguanozynową RNA
3. Procesy biologiczne z udziałem białka:
- inicjacja translacji w cytoplazmie
- kataboliczny proces jądrowo transkrybowanego mRNA (degradacja zależna od nonsensu (NMD))


###### Wnioski
Cztery białka (4/10) pełnią istotną rolę w procesie detoksykacji komórki i/lub w odpowiedzi komórki na stres (SPBC15D4.15, SPAC1002.07c, SPBC582.09, SPCC777.12c).
Trzy białka (3/10) biorą udział w procesach katabolicznych (SPAC1002.07c, SPBC1539.07c, SPAC16E8.15).
Dwa białka (2/10) biorą udział w procesie biosyntezy aminokwasów (SPAC19A8.15, SPAP7G5.04c).
Podsumowując, można stwierdzić, że funkcje poszczególnych białek są zbliżone do siebie. Warto zaznaczyć, że procesy, w których biorą udział te białka są kluczowe dla poprawnego funkcjonowania komórki, a ich zaburzenia mogą prowadzić do jej degradacji lub śmierci (np. wynikającej z "zagłodzenia" komórki z powodu brak tryptofanu w jej środowisku życia, przy jednoczesnej zaburzonej biosyntezie tego aminokwasu).

## Część II: Analiza ewolucyjna genu Creb1


In [ ]:
!pip install biopython

import polars as pl
import requests
from Bio import Entrez, SeqIO, AlignIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
import subprocess
import tempfile

!sudo apt update
!sudo apt install muscle



### Zadanie 1.
Przyjrzyj się rekordowi Q01147 dla czynnika transkrypcyjnego Creb1 w bazie Uniprot. Białko to posiada domenę bZIP umożliwiającą jego wiązanie do DNA

In [ ]:
# pobieramy zapis z uniprota
uniprot_id = "Q01147"
url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
import json
data = json.loads((Path(data_folder) / "uniprot" / f"{uniprot_id}.json").read_text())

print(data["proteinDescription"]["recommendedName"]["fullName"]["value"])
print(data["organism"]["scientificName"])

In [ ]:
data.keys()

### Zadanie 2.
W bazie Genbank znajdź sekwencje mRNA dojrzałych transkryptów genu Creb1 dla organizmów: Bombyx mori, Danio rerio, Oryctolagus cuniculus, Ovis aries, Sus scrofa (np. wpisując hasło “mrna complete cds” i dodając odpowiednie wartości w polach Gene Name oraz Organism w wyszukiwaniu zaawansowanym). Sekwencje powinny mieć długość między 800, a 1200 pz.


In [ ]:
#TODO - pass your mail !!!!
# Entrez.email = 'mb.strozyk@student.uw.edu.pl'


def fetch_mrna_gen(organism, gen='Creb1'):
  '''Returns mRNA sequences for given organism and gen from Genbank'''
  # ask for base
  query = (
      f'"{gen}"[Gene] AND "{organism}"[Organism] '
      f'AND mRNA[Filter] '
      f'AND 800:1200[SLEN]'
  )
  # entrez search engine
  search = Entrez.esearch(db="nucleotide", term=query)
  ids = Entrez.read(search)["IdList"]

  records = []
  for i in ids:
      handle = Entrez.efetch(
          db="nucleotide",
          id=i,
          rettype="gb",
          retmode="text"
        )
      records.append(SeqIO.read(handle, "genbank"))
  return records


In [ ]:
organisms = [
    "Bombyx mori",
    "Danio rerio",
    "Oryctolagus cuniculus",
    "Ovis aries",
    "Sus scrofa"
]

org_seq_map = dict()
for org in organisms:
    input_file = Path(data_folder) / "genbank" / f"{org.replace(' ', '_')}.gb"
    org_seq_map[org] = [] if org == "Bombyx mori" else list(SeqIO.parse(input_file, "genbank"))


In [ ]:
org_seq_map["Oryctolagus cuniculus"]

In [ ]:
for org, records in org_seq_map.items():
  print(org,records)


### Zadanie 3.
Rekordy w GenBank posiadają informację o współrzędnych sekwencji kodującej (CDS) oraz o sekwencji aminokwasowej otrzymywanej w wyniku translacji. Wykonaj uliniowienie (Multiple sequence alignment) kodowanych sekwencji aminokwasowych i wskaż na nim domenę bZIP - proszę o wklejenie wyniku w formie tekstowej lub zrzutu ekranu.


In [ ]:
def extract_coding_sequence(record):
    for f in record.features:
        if f.type == "CDS" and "translation" in f.qualifiers:
          # We return special string type that is IMMUTABLE
            return Seq(f.qualifiers["translation"][0])
    raise ValueError("No CDS translation found")

def collect_coding_sequences(org_seq_map):
    sequences = []

    for org, records in org_seq_map.items():
        for r in records:
            try:
                seq = extract_coding_sequence(r)
                sequences.append(
                    SeqRecord(
                        seq,
                        id=r.id,
                        description=org
                    )
                )
            except ValueError:
                continue

    return sequences


def align_sequences(
    seq_records,
    aligner="muscle",
    verbose=True
):
    """
    Takes a list of SeqRecord objects,
    performs MSA,
    """

    with tempfile.TemporaryDirectory() as tmpdir:
        fasta_in = f"{tmpdir}/input.fasta"
        fasta_out = f"{tmpdir}/aligned.fasta"

        if verbose:
            print(f"[MSA] Writing {len(seq_records)} sequences")

        SeqIO.write(seq_records, fasta_in, "fasta")

        if verbose:
            print(f"[MSA] Running {aligner}")

        if aligner == "muscle":
            subprocess.run(
                ["muscle", "-in", fasta_in, "-out", fasta_out],
                check=True
            )

        else:
            raise ValueError("Unsupported aligner")

        if verbose:
            print("[MSA] Reading alignment")

        alignment = AlignIO.read(fasta_out, "fasta")

    return alignment



In [ ]:
proteins = collect_coding_sequences(org_seq_map)

MSA_aligment  = align_sequences(proteins)

MSA_aligment

### Zadanie 4. Tabela aminokwasów
Stwórz tabele z poszczególnymi aminokwasami na kolejnych pozycjach otrzymanego uliniowienia.

In [ ]:
def align_to_polars(
    alignment,
    verbose=True
):
    if verbose:
        print("[MSA] Converting to Polars DataFrame")

    df = pl.DataFrame({
        "id": [rec.id for rec in alignment],
        **{
            f"pos_{i}": [rec.seq[i] for rec in alignment]
            for i in range(alignment.get_alignment_length())
        }
    })
    return df

In [ ]:
polar_alignment = align_to_polars(MSA_aligment)
polar_alignment

### Zadnie 5. Tabela kodonów
Stwórz analogiczną tabelę w której aminokwasy są zastąpione kodującymi je kodonami z rekordów z punktu 2 (tymi wewnątrz sekwencji kodującej).

In [ ]:
def align_to_codontable(alignment, org_seq_map, verbose=True):

    if verbose:
        print("[CodonTable] Building codon table")
    codon_data = {"id": [rec.id for rec in alignment]}
    seq_map = {}

    for org, records in org_seq_map.items():
        for r in records:
            seq_map[r.id] = r.seq

    for pos in range(alignment.get_alignment_length()):
        codon_column = []

        for rec in alignment:
            aa = rec.seq[pos]
            cds_seq = seq_map.get(rec.id)

            if cds_seq is None:
                codon_column.append('???')
                continue

            aa_index = sum(1 for c in rec.seq[:pos] if c != '-')
            start = aa_index * 3
            codon = cds_seq[start:start+3]

            if aa == '-':
                codon_column.append('---')
            else:
                codon_column.append(str(codon))

        codon_data[f"pos_{pos}"] = codon_column

    df_codons = pl.DataFrame(codon_data)
    if verbose:
        print("[CodonTable] Codon table done")
    return df_codons

In [ ]:
# tabela kodonów
df_codons = align_to_codontable(MSA_aligment , org_seq_map)
df_codons

### Zadanie 6. Analiza zmienności sekwencji aminokwasowej

#### Przygotowanie danych do analizy
Dla uproszczenia usuń kolumny zawierające insercje/delecje.



In [ ]:
# funkcja uniwersalna (dla tabeli aminokwasów i tabeli kodonów)
# usuwa kolumny zawierające insercje/delecje

def remove_gaps(df, gap_values=('-', '---'), id_col="id", verbose=True):

    cols_to_keep = [id_col]

    for col in df.columns:
        if col == id_col:
            continue
        has_gap = False
        for gap in gap_values:
            if (df[col] == gap).any():
                has_gap = True
                break
        if not has_gap:
            cols_to_keep.append(col)

    if verbose:
        removed = len(df.columns) - len(cols_to_keep)
        print(f"[RemoveGaps] Removed {removed} columns containing gaps")

    return df.select(cols_to_keep)

In [ ]:
# tabela aminokwasów bez kolumn zawierających insercje/delecje
df_aminoacids_nogaps = remove_gaps(polar_alignment)
df_aminoacids_nogaps

In [ ]:
# funkcja uniwersalna (dla tabeli aminokwasów i tabeli kodonów)
# oblicza jaki odsetek pozycji
# a) nie zawiera mutacji [no_mutation]
# b) zawiera pojedynczą mutację [single_mutation]
# c) zawiera więcej niż jedną mutację [multiple_mutations]

def variation_summary(df, cols=None, id_col="id"):
    if cols is None:
        cols = [c for c in df.columns if c != id_col]

    no_mut = 0
    single_mut = 0
    multi_mut = 0

    for col in cols:
        unique_vals = set(df[col].to_list())
        if len(unique_vals) == 1:
            no_mut += 1
        elif len(unique_vals) == 2:
            single_mut += 1
        else:
            multi_mut += 1

    total = len(cols)
    return {
        "no_mutation": round(no_mut / total, 3),
        "single_mutation": round(single_mut / total, 3),
        "multiple_mutations": round(multi_mut / total, 3)
    }

#### Pobranie zakresu domeny bZIP

In [ ]:
# zakres domeny bZIP

def get_bzip_range(uniprot_data):
    for f in uniprot_data["features"]:
        if f["type"] == "Domain" and f.get("description") == "bZIP":
            start = f["location"]["start"]["value"]
            end = f["location"]["end"]["value"]
            return start, end
    return None

In [ ]:
bzip_range = get_bzip_range(data)
print("bZIP range:", bzip_range)

#### Mapowanie domeny bZIP na tabelę aminokwasów (lub kodonów)

In [ ]:
# funkcja uniwersalna (dla tabeli aminokwasów i tabeli kodonów)
# mapowanie domeny bZIP na tabelę aminokwasów lub tabelę kodonów

def map_domain_to_table(df, domain_range, id_col="id"):

    if domain_range is None:
        # jeśli domena nieznana, wszystkie kolumny traktujemy jako "outside"
        domain_cols = []
        outside_cols = [c for c in df.columns if c != id_col]
        return domain_cols, outside_cols

    start, end = domain_range
    # 1-based -> 0-based dla kolumn pos_*
    domain_cols = [f"pos_{i-1}" for i in range(start, end+1) if f"pos_{i-1}" in df.columns]

    outside_cols = [c for c in df.columns if c not in domain_cols and c != id_col]

    return domain_cols, outside_cols

#### Analiza zmienności sekwencji (tabela aminokwasów)
Sprawdź jaki odsetek pozycji:
* a) nie zawiera mutacji,
* b) zawiera pojedynczą mutację (np. Bombyx: “V”, a pozostałe organizmy “T”), a jaki
* c) zawiera więcej niż jedną mutację.

Jak te proporcje wyglądają przy podziale na pozycje wewnątrz domeny bZIP i poza nią?

In [ ]:
# analiza zmienności sekwencji dla tabeli aminokwasów
aminoacids_summary = variation_summary(df_aminoacids_nogaps)
aminoacids_summary

In [ ]:
# proporcje mutacji przy podziale na pozycje wewnątrz domeny bZIP i poza nią

bzip_cols, outside_cols = map_domain_to_table(df_aminoacids_nogaps, bzip_range)

summary_bzip = variation_summary(df_aminoacids_nogaps, cols=bzip_cols)
summary_outside = variation_summary(df_aminoacids_nogaps, cols=outside_cols)

print("bZIP:", summary_bzip)
print("Outside bZIP:", summary_outside)


### Zadanie 7. Analiza zmienności sekwencji nukleotydowej
Wykonaj to samo zadanie co powyżej ale w odniesieniu do kodonów na pozycjach o
niezmutowanych aminokwasach, również z podziałem na pozycje wewnątrz domeny bZIP i
poza nią.

#### Przygotowanie danych do analizy

In [ ]:
# tabela kodonów bez kolumn zawierających insercje/delecje
df_codons_nogaps = remove_gaps(df_codons)
df_codons_nogaps

#### Analiza zmienności sekwencji (tabela kodonów)

In [ ]:
# analiza zmienności sekwencji dla tabeli kodonów
summary_codons = variation_summary(df_codons_nogaps)
print(summary_codons)


In [ ]:
# proporcje przy podziale na pozycje wewnątrz domeny bZIP i poza nią

bzip_cols, outside_cols = map_domain_to_table(df_codons_nogaps, bzip_range)

summary_bzip = variation_summary(df_codons_nogaps, cols=bzip_cols)
summary_outside = variation_summary(df_codons_nogaps, cols=outside_cols)

print("bZIP:", summary_bzip)
print("Outside bZIP:", summary_outside)